In [2]:
import torch
import torch.nn as nn
import pandas as pd

In [4]:
from torch.utils.data import TensorDataset

data = pd.read_csv("water_potability.csv")
data.head()

,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,0.587349,0.577747,0.386298,0.568199,0.647347,0.292985,0.654522,0.795029,0.630115,0
1,0.643654,0.441300,0.314381,0.439304,0.514545,0.356685,0.377248,0.202914,0.520358,0
2,0.388934,0.470876,0.506122,0.524364,0.561537,0.142913,0.249922,0.401487,0.219973,0
3,0.725820,0.715942,0.506141,0.521683,0.751819,0.148683,0.467200,0.658678,0.242428,0
4,0.610517,0.532588,0.237701,0.270288,0.495155,0.494792,0.409721,0.469762,0.585049,0


In [5]:
features = data.iloc[:, 1:-1]
X = features.to_numpy()

y = data.iloc[:, -1]
y = y.to_numpy()

In [10]:
dataset = TensorDataset(torch.Tensor(X).float(), torch.tensor(y).float())
# Access individual element
input_sample, label_sample = dataset[0]
print("input sample:", input_sample)
print("label sample:", label_sample)

input sample: tensor([0.5777, 0.3863, 0.5682, 0.6473, 0.2930, 0.6545, 0.7950, 0.6301])
label sample: tensor(0.)


In [13]:
from torch.utils.data import DataLoader

batch_size = 4
shuffle = True
## 1 Epoch is a full pass through the training dataloader
# # Generalization:   model performs well with unseen data

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [14]:
for batch_inputs, batch_labels in dataloader:
    print("batch_inputs:", batch_inputs)
    print("batch_labels:", batch_labels)

batch_inputs: tensor([[0.5757, 0.6259, 0.5942, 0.6187, 0.6661, 0.6154, 0.4697, 0.5838],
        [0.4607, 0.3015, 0.5061, 0.5713, 0.2262, 0.3810, 0.3384, 0.4350],
        [0.3167, 0.1970, 0.4615, 0.6867, 0.4165, 0.2221, 0.5636, 0.5227],
        [0.6032, 0.5183, 0.4231, 0.5504, 0.4411, 0.4365, 0.6000, 0.4632]])
batch_labels: tensor([0., 1., 0., 0.])
batch_inputs: tensor([[0.4856, 0.4955, 0.4891, 0.5406, 0.4470, 0.6922, 0.4537, 0.6063],
        [0.4729, 0.4263, 0.4647, 0.5603, 0.5415, 0.4504, 0.4967, 0.4927],
        [0.7761, 0.6907, 0.1546, 0.4452, 0.4924, 0.4475, 0.4916, 0.8567],
        [0.4601, 0.2839, 0.3345, 0.5338, 0.3945, 0.5573, 0.4861, 0.5865]])
batch_labels: tensor([0., 0., 0., 0.])
batch_inputs: tensor([[0.4099, 0.3435, 0.5117, 0.5479, 0.3445, 0.6552, 0.3272, 0.2420],
        [0.5403, 0.3595, 0.3849, 0.5237, 0.4605, 0.5572, 0.3775, 0.4480],
        [0.4965, 0.3545, 0.4370, 0.5776, 0.4977, 0.6325, 0.3841, 0.6195],
        [0.5203, 0.8489, 0.6624, 0.6003, 0.1981, 0.6679, 0.4773,

In [16]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
 
torch.manual_seed(42)
 
# ---------------------------------------------------------------------
# 1. Load and split the data
# ---------------------------------------------------------------------
df = pd.read_csv("water_potability.csv")
 
feature_cols = [c for c in df.columns if c != "Potability"]
X = df[feature_cols].values.astype(np.float32)
y = df["Potability"].values.astype(np.float32)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
print(f"Train samples: {len(X_train)}  |  Test samples: {len(X_test)}")
print(f"Features: {feature_cols}")
 
# Convert to tensors up front -- this is what makes TensorDataset viable:
# all the data already exists as tensors in memory, nothing needs to be
# loaded/decoded lazily per sample.
X_train_t = torch.from_numpy(X_train)
y_train_t = torch.from_numpy(y_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test)
 
 
# ---------------------------------------------------------------------
# 2a. Custom Dataset (the pattern for when __getitem__ needs real work)
# ---------------------------------------------------------------------
class WaterDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
 
    def __len__(self):
        return len(self.X)
 
    def __getitem__(self, idx):
        # In a real "expensive" dataset, this is where you'd read a file
        # from disk, decode an image, apply augmentation, etc. Here it's
        # trivial since the data's already tensors -- shown for the pattern.
        return self.X[idx], self.y[idx]
 
 
# ---------------------------------------------------------------------
# 2b. TensorDataset -- the equivalent, built-in shortcut
# ---------------------------------------------------------------------
# Since X_train_t / y_train_t are already full tensors in memory,
# TensorDataset(X, y) does exactly what WaterDataset does above, with
# no custom code needed. This is the one we'll actually use.
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
 
# Sanity check: confirm both approaches give identical samples
custom_ds = WaterDataset(X_train_t, y_train_t)
assert torch.equal(custom_ds[0][0], train_ds[0][0])
assert torch.equal(custom_ds[0][1], train_ds[0][1])
print("Custom Dataset and TensorDataset produce identical samples: OK")
 
 
# ---------------------------------------------------------------------
# 3. DataLoaders
# ---------------------------------------------------------------------
BATCH_SIZE = 32
 
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
# shuffle=True on train: reshuffles sample order every epoch, prevents the
#   model from learning anything spurious about row order.
# shuffle=False on test: no need to shuffle when we're just evaluating,
#   and keeping order fixed makes results easier to inspect/debug.
 
 
# ---------------------------------------------------------------------
# 4. Model
# ---------------------------------------------------------------------
class WaterClassifier(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),   # single logit output -- raw score, no sigmoid here
        )
 
    def forward(self, x):
        return self.net(x).squeeze(-1)   # (batch, 1) -> (batch,)
 
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WaterClassifier(in_features=X_train.shape[1]).to(device)
 
# BCEWithLogitsLoss combines a sigmoid + binary cross-entropy in one op.
# Numerically more stable than manually doing sigmoid() then BCELoss().
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
 
 
# ---------------------------------------------------------------------
# 5. Training loop
# ---------------------------------------------------------------------
def evaluate(loader):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
 
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == yb).sum().item()
            total += xb.size(0)
    model.train()
    return total_loss / total, correct / total
 
 
EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
 
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
 
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item() * xb.size(0)
 
    train_loss = running_loss / len(train_ds)
 
    if epoch % 5 == 0 or epoch == 1:
        test_loss, test_acc = evaluate(test_loader)
        print(f"Epoch {epoch:3d} | train_loss {train_loss:.4f} | "
              f"test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")
 
# ---------------------------------------------------------------------
# 6. Final evaluation
# ---------------------------------------------------------------------
final_loss, final_acc = evaluate(test_loader)
print(f"\nFinal test accuracy: {final_acc:.4f}")
 
baseline_acc = max((y_test == 0).mean(), (y_test == 1).mean())
print(f"Baseline (always predict majority class): {baseline_acc:.4f}")

Train samples: 1608  |  Test samples: 403
Features: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']
Custom Dataset and TensorDataset produce identical samples: OK
Epoch   1 | train_loss 0.6795 | test_loss 0.6763 | test_acc 0.5955
Epoch   5 | train_loss 0.6732 | test_loss 0.6750 | test_acc 0.5955
Epoch  10 | train_loss 0.6696 | test_loss 0.6735 | test_acc 0.5955
Epoch  15 | train_loss 0.6628 | test_loss 0.6701 | test_acc 0.6055
Epoch  20 | train_loss 0.6552 | test_loss 0.6685 | test_acc 0.6179
Epoch  25 | train_loss 0.6472 | test_loss 0.6626 | test_acc 0.6228
Epoch  30 | train_loss 0.6393 | test_loss 0.6562 | test_acc 0.6377
Epoch  35 | train_loss 0.6305 | test_loss 0.6487 | test_acc 0.6402
Epoch  40 | train_loss 0.6205 | test_loss 0.6399 | test_acc 0.6576
Epoch  45 | train_loss 0.6108 | test_loss 0.6351 | test_acc 0.6700
Epoch  50 | train_loss 0.6055 | test_loss 0.6328 | test_acc 0.6749
Epoch  55 | train_loss 0.59